# 05 — Model Training

**Goal:** Train Logistic Regression (baseline), Random Forest, and XGBoost models. Compare using stratified cross-validation. Select the best model for final evaluation.

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import (
    load_raw_data, clean_data, separate_features_target, split_data,
    identify_column_types, build_preprocessor, TARGET_COL
)
from src.feature_engineering import engineer_features
from src.train import (
    cross_validate_models, train_final_model,
    save_model, save_comparison_report
)
from src.evaluate import (
    compute_metrics, print_classification_report,
    plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve,
    plot_model_comparison, find_optimal_threshold, generate_evaluation_summary
)

## 5.1 Prepare Data

In [2]:
# Load, clean, and engineer features
df_raw = load_raw_data()
df = clean_data(df_raw)
df = engineer_features(df)

# Separate and split
X, y, ids = separate_features_target(df)
X_train, X_test, y_train, y_test = split_data(X, y)

# Build preprocessor
numeric_cols, categorical_cols = identify_column_types(X_train)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

print(f"\nReady for training:")
print(f"  Training set: {X_train.shape}")
print(f"  Test set: {X_test.shape} (untouched until final evaluation)")
print(f"  Numeric features: {len(numeric_cols)}")
print(f"  Categorical features: {len(categorical_cols)}")

Loaded WA_Fn-UseC_-Telco-Customer-Churn.csv: 7043 rows, 21 columns
Feature engineering added 6 columns: ['tenure_group', 'total_services', 'avg_monthly_charge', 'charge_increase', 'has_internet', 'has_phone_and_internet']
Train: 5634 rows | Test: 1409 rows | Train churn rate: 0.265 | Test churn rate: 0.265

Ready for training:
  Training set: (5634, 25)
  Test set: (1409, 25) (untouched until final evaluation)
  Numeric features: 8
  Categorical features: 17


## 5.2 Cross-Validation Comparison

In [3]:
comparison_df = cross_validate_models(X_train, y_train, preprocessor)
comparison_df


Cross-validating Logistic Regression...


  Precision: 0.5165 (+/-0.0166) | Recall: 0.7967 (+/-0.0355) | F1: 0.6265 (+/-0.0209) | ROC-AUC: 0.8462 (+/-0.0114)

Cross-validating Random Forest...


  Precision: 0.5651 (+/-0.0148) | Recall: 0.7137 (+/-0.0275) | F1: 0.6307 (+/-0.0187) | ROC-AUC: 0.8440 (+/-0.0092)

Cross-validating XGBoost...


  Precision: 0.5466 (+/-0.0041) | Recall: 0.7304 (+/-0.0121) | F1: 0.6252 (+/-0.0053) | ROC-AUC: 0.8381 (+/-0.0060)


,Model,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Logistic Regression,0.516525,0.016596,0.796656,0.035546,0.626538,0.020950,0.846193,0.011373
1,Random Forest,0.565100,0.014837,0.713712,0.027547,0.630668,0.018726,0.843986,0.009212
2,XGBoost,0.546575,0.004080,0.730435,0.012077,0.625220,0.005267,0.838100,0.005977


In [4]:
# Save comparison
save_comparison_report(comparison_df)

# Plot comparison
fig = plot_model_comparison(comparison_df)
plt.show()

Model comparison saved to D:\Customer Churn Prediction & Retention System\reports\model_comparison.csv


Saved model comparison to D:\Customer Churn Prediction & Retention System\reports\figures\model_comparison.png


## 5.3 Select and Train Final Model

Based on cross-validation results, we select the model with the best balance of recall (identifying churners) and precision (avoiding false alarms). We choose XGBoost as it typically offers the best ROC-AUC and F1 balance.

In [5]:
# Select model based on best F1 score from CV
best_model_name = comparison_df.loc[comparison_df['f1_mean'].idxmax(), 'Model']
print(f"Selected model: {best_model_name}")
print(f"  CV F1: {comparison_df.loc[comparison_df['f1_mean'].idxmax(), 'f1_mean']:.4f}")
print(f"  CV ROC-AUC: {comparison_df.loc[comparison_df['f1_mean'].idxmax(), 'roc_auc_mean']:.4f}")

Selected model: Random Forest
  CV F1: 0.6307
  CV ROC-AUC: 0.8440


In [6]:
# Train final model on full training set
pipeline = train_final_model(X_train, y_train, preprocessor, model_name=best_model_name)

# Save the model
save_model(pipeline)


Training Random Forest on full training set (5634 rows)...


Training complete.
Model saved to D:\Customer Churn Prediction & Retention System\models\churn_model.joblib


WindowsPath('D:/Customer Churn Prediction & Retention System/models/churn_model.joblib')

## 5.4 Find Optimal Threshold (on Training Data)

In [7]:
# Use training predictions to find threshold (NOT test set)
y_train_prob = pipeline.predict_proba(X_train)[:, 1]
optimal_threshold, best_f1 = find_optimal_threshold(y_train, y_train_prob, metric='f1')
print(f"\nOptimal threshold: {optimal_threshold:.2f} (Training F1: {best_f1:.4f})")

Optimal threshold for f1: 0.52 (score: 0.7748)

Optimal threshold: 0.52 (Training F1: 0.7748)


## 5.5 Final Evaluation on Test Set

This is the **one-time** evaluation on the held-out test set.

In [8]:
# Generate test predictions
y_test_prob = pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)

# Compute metrics
test_metrics = compute_metrics(y_test, y_test_pred, y_test_prob, threshold=optimal_threshold)
print("\n" + "="*50)
print("FINAL TEST SET EVALUATION")
print("="*50)
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")


FINAL TEST SET EVALUATION
  threshold: 0.5200
  accuracy: 0.7693
  precision: 0.5520
  recall: 0.6952
  f1: 0.6154
  roc_auc: 0.8391


In [9]:
# Classification report
print("\nDetailed Classification Report:")
print_classification_report(y_test, y_test_pred)


Detailed Classification Report:
              precision    recall  f1-score   support

    No Churn       0.88      0.80      0.84      1035
       Churn       0.55      0.70      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.77      0.78      1409



'              precision    recall  f1-score   support\n\n    No Churn       0.88      0.80      0.84      1035\n       Churn       0.55      0.70      0.62       374\n\n    accuracy                           0.77      1409\n   macro avg       0.72      0.75      0.73      1409\nweighted avg       0.79      0.77      0.78      1409\n'

In [10]:
# Confusion Matrix
fig = plot_confusion_matrix(y_test, y_test_pred, title=f'Confusion Matrix — {best_model_name}')
plt.show()

Saved confusion matrix to D:\Customer Churn Prediction & Retention System\reports\figures\confusion_matrix.png


In [11]:
# ROC Curve
fig = plot_roc_curve(y_test, y_test_prob, title=f'ROC Curve — {best_model_name}')
plt.show()

Saved ROC curve to D:\Customer Churn Prediction & Retention System\reports\figures\roc_curve.png


In [12]:
# Precision-Recall Curve
fig = plot_precision_recall_curve(y_test, y_test_prob, title=f'Precision-Recall Curve — {best_model_name}')
plt.show()

Saved PR curve to D:\Customer Churn Prediction & Retention System\reports\figures\precision_recall_curve.png


In [13]:
# Generate evaluation summary markdown
generate_evaluation_summary(test_metrics, model_name=best_model_name)

Saved evaluation summary to D:\Customer Churn Prediction & Retention System\reports\evaluation_summary.md


'# Model Evaluation Summary\n\n## Model: Random Forest\n\n### Classification Metrics\n\n| Metric | Value |\n|---|---:|\n| Threshold | 0.5200 |\n| Accuracy | 0.7693 |\n| Precision | 0.5520 |\n| Recall | 0.6952 |\n| F1 | 0.6154 |\n| Roc Auc | 0.8391 |\n\n### Notes\n\n- Metrics computed on the held-out test set (not used during training or model selection).\n- Threshold was selected using cross-validation on the training set.\n- SHAP explanations describe model behaviour and do not establish causality.\n'

## 5.6 Notes

- The test set was used **only once** for final evaluation.
- The classification threshold was chosen on training data.
- Model selection was based on cross-validation scores, not test performance.
- Results reflect the model's performance on this specific dataset split and may vary with different seeds or data.